# Wav2Vec 2.0

A self-supervised model that learns speech representations from **raw, unlabeled audio**,
then fine-tunes with a thin head for downstream tasks (most famously ASR via CTC).

**Domain:** Speech & Audio  ·  **runnable:** yes

## 1. What & Why

**What it is.** Wav2Vec 2.0 (Baevski et al., 2020, Meta AI) is a transformer that takes a
**16 kHz waveform** and produces ~50 contextual feature vectors per second of audio. Its trick
is *self-supervised pretraining*: it learns from thousands of hours of **unlabeled** speech by
masking spans of the audio and solving a contrastive task — pick the true quantized latent for
each masked step out of a set of distractors. Only afterward do you bolt on a task head and
fine-tune on a (often tiny) labeled dataset.

**The problem it solves.** Labeled speech is expensive; raw audio is nearly free. Before
wav2vec 2.0, a good ASR model needed hundreds–thousands of hours of *transcribed* audio.
Pretraining on unlabeled audio lets you fine-tune a strong recognizer with **as little as 10
minutes to 100 hours** of labels and still beat older fully-supervised systems. It moved speech
into the same "pretrain once, fine-tune cheaply" paradigm that BERT brought to NLP.

**When to reach for it.**
- You have a custom domain (medical, call-center, a low-resource language) with **little
  labeled data** but access to in-domain audio — fine-tune wav2vec 2.0.
- You want a strong **frozen feature extractor** for emotion recognition, speaker ID, keyword
  spotting, or audio classification.
- You want a self-hostable, open-weights ASR baseline (`wav2vec2-base-960h`,
  `wav2vec2-large-960h-lv60-self`, or XLSR for multilingual).

**When not to.** If you just want the best turnkey transcription with punctuation, casing, and
language ID out of the box, **Whisper** is usually less work and more robust to noise. Wav2Vec 2.0's
vanilla CTC output is uppercase, unpunctuated, and English-only unless you chose a multilingual
checkpoint.

## 2. Mental Model

Think of it as a **two-stage pipeline that learned to listen before it learned to read**:

```
raw 16 kHz waveform
   │
   ▼  CNN feature encoder  (7 conv layers, total stride 320)
local latents  z₁ z₂ … zT      ← one vector per ~20 ms frame  (≈ 50 / second)
   │                 │
   │                 ├─► quantizer  →  q (discrete codebook targets)   [pretraining only]
   ▼
Transformer (context network, 12 or 24 layers)
   │
   ▼
contextual reps  c₁ c₂ … cT
   │
   ▼  task head
   ├─ pretraining:  contrastive loss — match cₜ at MASKED steps to its own qₜ vs distractors
   └─ fine-tuning:  linear → CTC over characters  (or a pooled classifier for audio tasks)
```

Two mantras:
1. **"BERT for audio."** Mask spans of the latent sequence, make the transformer predict the
   masked content — but the prediction target is a *learned discrete code* (the quantizer), not a
   word, because audio has no tokens.
2. **"The CNN sets the clock."** The conv stack downsamples 16,000 samples/s down to ~50
   frames/s. Everything downstream — CTC alignment, the length of your logits — runs on that
   50 Hz grid (one frame ≈ 20 ms).

## 3. Key Concepts

- **Feature encoder (CNN).** 7 temporal conv layers turn the waveform into latents `z`. Total
  stride is **320 samples** → at 16 kHz that's one frame every **20 ms** (~50 fps). Receptive
  field ≈ 400 samples (25 ms). In `transformers` it's usually **frozen** during fine-tuning.
- **Context network (Transformer).** 12 layers (base, 768-d) or 24 (large, 1024-d) add
  bidirectional context, producing `c`. This is where most of the parameters and the "understanding" live.
- **Quantization / codebooks.** A Gumbel-softmax product-quantizer maps each latent to a
  discrete code from learnable codebooks. These discrete codes are the **prediction targets** for
  pretraining (you can't predict a raw continuous vector contrastively as cleanly).
- **Masking.** During pretraining, spans of latent timesteps are masked (≈ 6.5% of starting
  indices, span length 10) — like BERT masking but on audio frames.
- **Contrastive + diversity loss.** For each masked step, identify the true quantized code vs K
  distractors (InfoNCE). A diversity term keeps all codebook entries in use.
- **CTC fine-tuning.** Add a linear layer to a character vocabulary plus a **blank** token, train
  with **Connectionist Temporal Classification** — no frame-level alignment needed. Decode greedily
  (collapse repeats, drop blanks) or with a KenLM beam-search language model for accuracy.
- **16 kHz mono, normalized.** The model expects single-channel 16 kHz float audio, typically
  zero-mean/unit-variance normalized (the `Wav2Vec2Processor` does this for you).
- **Checkpoints.** `base` (95M) vs `large` (317M); `-960h` = fine-tuned on 960 h LibriSpeech;
  `XLSR` / `mms` = cross-lingual, 53–1000+ languages.

## 4. Setup

Pretrained weights and CTC inference live in Hugging Face `transformers`. You need a backend
(`torch`), `transformers`, and an audio loader (`torchaudio` or `librosa`/`soundfile`).

```bash
%pip install torch transformers torchaudio soundfile
```

The two worked examples below are **pure-Python** (numpy only) and always run — they teach the
frame-rate math and CTC decoding that govern every wav2vec 2.0 model. The third example does a
**real forward pass** but is gated behind an env var because it downloads a ~360 MB checkpoint.

In [1]:
import numpy as np

# Versions are informational; the first two examples need only numpy.
try:
    import torch, transformers
    print("torch:", torch.__version__, "| transformers:", transformers.__version__)
except ImportError:
    print("torch/transformers not installed — pure-Python examples still run.")

/Users/danieldekerlegand/Development/ai-tutor/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.12.1 | transformers: 5.12.1


## 5. Worked Examples

1. **Frame-rate / output-length math** — why a 5-second clip becomes ~249 frames, and how to map
   a CTC frame index back to a timestamp. This is the single most useful number to keep in your head.
2. **Greedy CTC decoding** — collapse a frame-level argmax into text the way every wav2vec 2.0
   ASR model does at inference.
3. **Real inference** (gated) — load `wav2vec2-base-960h`, transcribe a tone/synthetic clip, and
   show the exact `Processor → model → argmax → decode` call shape.

In [2]:
# Example 1 — The CNN "clock": from samples to frames to seconds.
# The 7 conv layers of the feature encoder have these (kernel, stride) pairs:
CONV = [(10, 5), (3, 2), (3, 2), (3, 2), (3, 2), (2, 2), (2, 2)]

def out_len(n, kernel, stride):
    # PyTorch Conv1d length formula (no padding, dilation=1).
    return (n - kernel) // stride + 1

def frames_for(num_samples):
    n = num_samples
    for k, s in CONV:
        n = out_len(n, k, s)
    return n

SR = 16_000
total_stride = 1
for _, s in CONV:
    total_stride *= s
print("Total downsampling stride:", total_stride, "samples")
print("Frame rate:", SR / total_stride, "frames/sec  ->  one frame every",
      round(1000 * total_stride / SR, 1), "ms\n")

for secs in (1, 5, 10):
    n = secs * SR
    f = frames_for(n)
    print(f"{secs:>2}s audio = {n:>7} samples -> {f:>4} transformer frames "
          f"(~{f/secs:.0f} fps)")

# Map a CTC frame index back to a wall-clock timestamp.
frame_idx = 100
print(f"\nFrame {frame_idx} starts at ~{frame_idx * total_stride / SR:.2f}s")

Total downsampling stride: 320 samples
Frame rate: 50.0 frames/sec  ->  one frame every 20.0 ms

 1s audio =   16000 samples ->   49 transformer frames (~49 fps)
 5s audio =   80000 samples ->  249 transformer frames (~50 fps)
10s audio =  160000 samples ->  499 transformer frames (~50 fps)

Frame 100 starts at ~2.00s


In [3]:
# Example 2 — Greedy CTC decoding: turn per-frame predictions into a string.
# wav2vec 2.0 ASR emits, for each ~20ms frame, a distribution over a char vocab
# that includes a special BLANK. Greedy decode = argmax per frame, then:
#   (1) collapse consecutive duplicate labels, (2) remove blanks.
BLANK = "_"
vocab = [BLANK, "H", "E", "L", "O", " "]   # tiny toy vocabulary

# Pretend these are the per-frame argmax token ids for the word "HELLO".
# Note the repeats (the model fires the same char over several 20ms frames)
# and the blanks that SEPARATE the two L's so they don't collapse into one.
frame_ids = [0, 1, 1, 0, 2, 2, 2, 0, 3, 3, 0, 3, 0, 4, 4, 4, 0, 0]
frame_tokens = [vocab[i] for i in frame_ids]
print("Per-frame argmax:", "".join(frame_tokens))

def ctc_greedy_decode(tokens, blank=BLANK):
    out, prev = [], None
    for t in tokens:
        if t != prev and t != blank:   # new, non-blank symbol
            out.append(t)
        prev = t                       # blank also resets `prev`
    return "".join(out)

print("Decoded text   :", repr(ctc_greedy_decode(frame_tokens)))

# The blank between the two L's is essential. Without it they collapse:
no_blank = [vocab[i] for i in [1, 2, 3, 3, 4]]   # H E L L O, no separating blank
print("No-blank L L   :", repr(ctc_greedy_decode(no_blank)), "<- double letter lost")

Per-frame argmax: _HH_EEE_LL_L_OOO__
Decoded text   : 'HELLO'
No-blank L L   : 'HELO' <- double letter lost


In [4]:
# Example 3 — A real forward pass (gated: downloads ~360 MB).
# Set RUN_WAV2VEC2=1 to actually load wav2vec2-base-960h and transcribe.
import os

if os.getenv("RUN_WAV2VEC2"):
    import torch
    from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

    name = "facebook/wav2vec2-base-960h"
    processor = Wav2Vec2Processor.from_pretrained(name)
    model = Wav2Vec2ForCTC.from_pretrained(name).eval()

    # Synthetic 2s clip (silence will transcribe to ""; swap in real speech to see text).
    sr = 16_000
    speech = np.zeros(2 * sr, dtype=np.float32)

    inputs = processor(speech, sampling_rate=sr, return_tensors="pt")
    with torch.no_grad():
        logits = model(inputs.input_values).logits          # (1, frames, vocab)
    pred_ids = torch.argmax(logits, dim=-1)                  # greedy CTC
    text = processor.batch_decode(pred_ids)[0]
    print("logits shape:", tuple(logits.shape))
    print("transcript  :", repr(text))
else:
    print("RUN_WAV2VEC2 not set — skipping the ~360 MB download.")
    print("The call shape you'd run:")
    print("  processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h')")
    print("  model     = Wav2Vec2ForCTC.from_pretrained('facebook/wav2vec2-base-960h')")
    print("  logits    = model(processor(speech, sampling_rate=16000,")
    print("                              return_tensors='pt').input_values).logits")
    print("  text      = processor.batch_decode(logits.argmax(-1))[0]")

RUN_WAV2VEC2 not set — skipping the ~360 MB download.
The call shape you'd run:
  processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h')
  model     = Wav2Vec2ForCTC.from_pretrained('facebook/wav2vec2-base-960h')
  logits    = model(processor(speech, sampling_rate=16000,
                              return_tensors='pt').input_values).logits
  text      = processor.batch_decode(logits.argmax(-1))[0]


## 6. Gotchas & Pitfalls

- **Sample rate must be 16 kHz.** Feeding 8 kHz, 22.05 kHz, or 44.1 kHz audio silently wrecks
  accuracy — the CNN's strides assume 16 kHz. Resample first (`torchaudio.functional.resample`).
  (The multilingual MMS/XLSR models are also 16 kHz.)
- **Mono only, and normalize.** Pass single-channel float audio; let `Wav2Vec2Processor`
  zero-mean/unit-variance normalize. The `-960h` processors set `do_normalize=True`; skipping it
  hurts the large checkpoints especially.
- **`Wav2Vec2ForCTC` ≠ a pretrained-only checkpoint.** `facebook/wav2vec2-base` (no `-960h`) has
  **no CTC head** — loading it into `ForCTC` gives a randomly-initialized head that outputs garbage.
  Use a `-960h`/`-self` checkpoint for ready-to-go transcription, or fine-tune the base yourself.
- **Output is UPPERCASE, no punctuation, English-only.** Vanilla CTC has a character vocab with no
  casing or punctuation. If you need readable text, add a language model / post-processor or use
  Whisper. For other languages you must pick XLSR/MMS, not `-960h`.
- **Greedy decode leaves accuracy on the table.** A KenLM n-gram + beam search
  (`Wav2Vec2ProcessorWithLM`, the `pyctcdecode` path) typically cuts WER noticeably. Greedy is fine
  for a demo, not for production.
- **Don't collapse the blank away too early.** The blank token is what lets CTC emit real double
  letters ("LL", "OO"). Custom decoders that treat blank as just another char will merge them.
- **Freeze the feature encoder when fine-tuning.** Call `model.freeze_feature_encoder()`. Training
  the CNN on small data is unstable and rarely helps; the standard recipe freezes it.
- **Long audio = memory blowup.** Self-attention is O(n²) in frames, and a 50 fps grid grows fast.
  Chunk audio (e.g. 20–30 s windows with small overlap) for long recordings.
- **CTC training instability.** Mind the `pad_token_id`/blank index, use a warmup schedule, and
  expect loss to plateau then drop. A wrong blank index produces empty transcripts.

## 7. When to Use vs Alternatives

| Option | Strengths | Weaknesses | Reach for it when |
|---|---|---|---|
| **Wav2Vec 2.0 (CTC)** | Open weights, self-hostable, excellent for **fine-tuning on low-resource / in-domain** data; great frozen features | Raw output is uppercase/unpunctuated/English (unless XLSR/MMS); greedy WER needs an LM; less robust to noise than Whisper | You have in-domain audio + few labels, or need an embeddings backbone |
| **Whisper** (OpenAI) | Turnkey multilingual transcription **with punctuation & casing**, robust to noise/accents, built-in language ID | Heavier, slower, hallucinates on silence/long gaps; harder to fine-tune cheaply | You want the best out-of-the-box transcript with minimal effort |
| **HuBERT / WavLM** | Same architecture family; **WavLM** is stronger for speaker/diarization & noisy-overlap tasks | Same 16 kHz/CTC caveats as wav2vec | Speaker ID, diarization, paralinguistics — especially WavLM |
| **XLSR-53 / MMS** | Wav2Vec pretrained **cross-lingually** (53 → 1000+ languages) | Per-language fine-tuning still needed for best ASR | Non-English / low-resource languages |
| **NVIDIA NeMo (Conformer/Citrinet)** | Production streaming ASR, strong toolkit, RNN-T options | Heavier framework, GPU-centric | You need streaming / a full production ASR stack |
| **Classic features (MFCC + HMM/GMM)** | Tiny, CPU-cheap, interpretable | Far worse accuracy | Ultra-constrained or legacy systems |

**Rule of thumb:** *Custom domain + limited labels, or you need open self-hosted weights / audio
embeddings* → wav2vec 2.0 (or WavLM). *Just want a great transcript with no training* → Whisper.

## 8. Resources

- **Paper — "wav2vec 2.0: A Framework for Self-Supervised Learning of Speech Representations"**
  (Baevski, Zhou, Mohamed, Auli, 2020): https://arxiv.org/abs/2006.11477
- **Hugging Face model docs (`Wav2Vec2`)**:
  https://huggingface.co/docs/transformers/model_doc/wav2vec2
- **HF blog — fine-tuning wav2vec 2.0 for ASR (with CTC, end-to-end walkthrough)**:
  https://huggingface.co/blog/fine-tune-wav2vec2-english
- **`facebook/wav2vec2-base-960h` model card** (ready-to-use English ASR):
  https://huggingface.co/facebook/wav2vec2-base-960h
- **fairseq original implementation**:
  https://github.com/facebookresearch/fairseq/tree/main/examples/wav2vec
- **CTC explainer (Distill — "Sequence Modeling with CTC")**:
  https://distill.pub/2017/ctc/
- **Cross-lingual: XLSR-53 paper** (https://arxiv.org/abs/2006.13979) and
  **MMS** (https://huggingface.co/docs/transformers/model_doc/mms) for multilingual speech.